# Predicting Influenza A Host Species from PB2 Sequences with ESM-2

**Task.** Given the amino-acid sequence of the influenza A polymerase basic protein 2
(PB2) segment, predict which host class the virus was isolated from: **human**,
**avian**, or **other mammal**.

**Why PB2.** PB2 is the classic host-range determinant in influenza A. Adaptation
mutations at PB2 residues 627 and 701 are among the best-characterised markers of avian
to mammalian host switching, so if any single segment carries host signal in its primary
sequence, it is this one. The dataset here is entirely H5N1, the subtype currently
circulating in North American dairy cattle.

**Approach.** ESM-2 is a protein language model — architecturally a BERT-style masked
language model, but trained on ~65M amino-acid sequences instead of text. It gives every
residue a 1280-dimensional contextual embedding. We use it three ways, in increasing
order of how much we let the model adapt:

| Stage | What trains | Question it answers |
|---|---|---|
| 1. Frozen embeddings + LR / random forest | classifier only | Do the pretrained representations already separate hosts? |
| 2. Frozen embeddings + MLP head | small MLP only | Does a nonlinear head extract more? |
| 3. Fine-tuned ESM-2 (last 3–8 layers) + deep head | ESM-2 upper layers + head | Does task-specific adaptation help? |

**Result:** it does *not* help. The frozen random forest approach wins with **90.97%** test
accuracy,  both the MLP head (84.11%) and end-to-end fine-tuning (89.10%).

---

## 0. Setup

The full pipeline needs a GPU — embedding extraction over 1602 sequences at ~759
residues each is the bottleneck. Fine-tuning the 650M-parameter checkpoint needs roughly
16GB of VRAM at `batch_size=4`. Both stages were originally run on Google Colab (T4/A100) and on
an HPC node.

In [ ]:
# !pip install fair-esm biopython umap-learn torch scikit-learn matplotlib seaborn pandas --quiet

In [2]:
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import esm
from Bio import SeqIO

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns
import umap

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

In [3]:
# Paths are relative to the repository root
DATA_DIR    = Path("../data")
RESULTS_DIR = Path("../results")

# The real GISAID sequences cannot be redistributed (see data/README.md).
# Set USE_SYNTHETIC_DATA = True to run the full pipeline on generated stand-in
# data instead. Generate it first with:
#     python scripts/generate_synthetic_data.py
#
# Disclaimer: Synthetic results are a smoke test that the code runs end to end. They are NOT
# the results reported in the README and say nothing about influenza biology.

USE_SYNTHETIC_DATA = False

if USE_SYNTHETIC_DATA:
    FASTA_PATH = DATA_DIR / "synthetic_pb2_sequences.fasta"
    print("WARNING: using SYNTHETIC data — results are not scientifically meaningful.")
else:
    FASTA_PATH = DATA_DIR / "all_pb2_sequences_labeled_balanced.fasta"

HOST_MAPPING = {"HOST_0": "human", "HOST_1": "avian", "HOST_2": "mammal"}

---

## 1. Data preparation

Sequences were downloaded from **GISAID EpiFlu** as three separate FASTA files, one per
host class, filtered to influenza A / H5N1, PB2 segment.

A GISAID header looks like this:

```
>SEQID|PB2|A/strain/name/here/2024|ISOLATE_ID|A_/_H5N1
```

| Field | Meaning |
|---|---|
| `SEQID` | sequence accession |
| `PB2` | segment (influenza A has 8; we use only this one) |
| `A/strain/name/here/2024` | standard influenza strain nomenclature |
| `ISOLATE_ID` | isolate accession |
| `A_/_H5N1` | type and subtype |

Because the host class is only recoverable from the strain-name field — and parsing
`dairy_cow` vs `chicken` vs a place name is error-prone — we append an explicit
`|HOST_n` tag to every header at download time, then concatenate.

In [ ]:
def add_host_label(fasta_file, output_file, host_type):
    """Append |HOST_{n} to every header. 0=human, 1=avian, 2=mammal."""
    with open(fasta_file) as infile, open(output_file, "w") as outfile:
        for line in infile:
            if line.startswith(">"):
                outfile.write(line.strip() + f"|HOST_{host_type}\n")
            else:
                outfile.write(line)


def concatenate_labeled(paths_by_host, output_path):
    """Merge the three per-host labeled FASTAs into one file."""
    records, counts = [], defaultdict(int)
    for path, host_code in paths_by_host.items():
        for record in SeqIO.parse(path, "fasta"):
            records.append(record)
            counts[host_code] += 1
    SeqIO.write(records, output_path, "fasta")
    for host_code, n in sorted(counts.items()):
        print(f"  HOST_{host_code}: {n} sequences")
    print(f"Total: {sum(counts.values())}")
    return records

### Class balancing

The raw pull was imbalanced across the three host classes. Rather than carry class
weights through every downstream model, we **downsample to the smallest class (534)** so
that chance accuracy is a clean 33.3% and the confusion matrices are directly readable.

This costs us training data, which matters — see the discussion in section 6.

In [ ]:
def downsample_fasta(input_fasta, output_fasta, target_count=534, random_seed=RANDOM_STATE):
    """Randomly downsample each host class to target_count sequences."""
    random.seed(random_seed)
    by_host = defaultdict(list)

    for record in SeqIO.parse(input_fasta, "fasta"):
        parts = record.description.split("|")
        if len(parts) >= 6:
            by_host[parts[5].strip()].append(record)

    print("Original distribution:")
    for host_code in sorted(by_host):
        print(f"  {host_code}: {len(by_host[host_code])}")

    balanced = []
    for host_code in sorted(by_host):
        seqs = by_host[host_code]
        balanced.extend(random.sample(seqs, target_count) if len(seqs) > target_count else seqs)

    random.shuffle(balanced)
    SeqIO.write(balanced, output_fasta, "fasta")
    print(f"\nBalanced dataset: {len(balanced)} sequences -> {output_fasta}")
    return balanced

In [ ]:
def parse_gisaid_fasta(fasta_path):
    """Parse the labeled FASTA into (seq_id, sequence) pairs plus a metadata frame."""
    sequences, metadata = [], []

    for record in SeqIO.parse(fasta_path, "fasta"):
        parts = record.description.split("|")
        seq_str = str(record.seq)
        if len(parts) < 6 or not seq_str:
            continue

        host_code = parts[5].strip()
        host_type = HOST_MAPPING.get(host_code)
        if host_type is None:
            continue

        sequences.append((record.id, seq_str))
        metadata.append({
            "seq_id": record.id,
            "epi_id": parts[0],
            "gene": parts[1],
            "strain": parts[2],
            "epi_isl": parts[3],
            "subtype": parts[4],
            "host_code": host_code,
            "host_type": host_type,
            "sequence_length": len(seq_str),
        })

    metadata_df = pd.DataFrame(metadata)
    print(f"Loaded {len(sequences)} sequences\n")
    print(metadata_df["host_type"].value_counts(), "\n")
    print(metadata_df.groupby("host_type")["sequence_length"].describe())
    return sequences, metadata_df


sequences, metadata_df = parse_gisaid_fasta(FASTA_PATH)

**A note on the length distribution.** All three classes centre on 759 residues, the
canonical PB2 length, but the minima differ sharply — human sequences go down to 32
residues, avian to 111, mammal to 401. Those short entries are partial submissions, not
biology. They are retained here (the original analysis did not filter them), but they are
a plausible source of label-correlated noise: if truncation rate differs by host class,
a classifier can learn submission practice rather than viral adaptation. Flagged as
future work in section 6.

---

## 2. ESM-2 embeddings

ESM-2 (`esm2_t33_650M_UR50D`) is a 33-layer transformer trained with a masked-language
objective on UniRef50. Each residue receives a 1280-dim contextual vector encoding
chemical and structural properties learned from evolutionary statistics.

To get one vector per *sequence* we **mean-pool** over residue embeddings from the final
layer, excluding the `<cls>` and `<eos>` special tokens.

In [ ]:
def load_esm2_model(model_name="esm2_t33_650M_UR50D"):
    print(f"Loading {model_name}...")
    model, alphabet = esm.pretrained.__dict__[model_name]()
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"Model loaded on {device}")
    return model, alphabet, device


model, alphabet, device = load_esm2_model()

In [ ]:
def extract_esm2_embeddings(sequences, model, alphabet, device,
                            batch_size=8, return_contacts=False):
    """Mean-pooled final-layer ESM-2 embedding per sequence.

    Set return_contacts=True only for the attention analysis in section 3 --
    the contact head is what causes CUDA OOM on longer sequences.
    """
    batch_converter = alphabet.get_batch_converter()
    sequence_embeddings = {}

    print(f"Extracting embeddings for {len(sequences)} sequences...")
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]
        _, _, batch_tokens = batch_converter(batch)
        batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
        batch_tokens = batch_tokens.to(device)

        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[33], return_contacts=return_contacts)

        token_representations = results["representations"][33]

        for j, (seq_id, _) in enumerate(batch):
            tokens_len = batch_lens[j].item()
            residue_emb = token_representations[j, 1:tokens_len - 1].cpu().numpy()
            sequence_embeddings[seq_id] = residue_emb.mean(0)

        del batch_tokens, results, token_representations
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (i + batch_size) % 400 == 0 or (i + batch_size) >= len(sequences):
            print(f"  {min(i + batch_size, len(sequences))}/{len(sequences)}")

    return sequence_embeddings


sequence_embeddings = extract_esm2_embeddings(sequences, model, alphabet, device, batch_size=8)

### UMAP: is there structure before we train anything?

Projecting the frozen 1280-dim embeddings to 2D with cosine-metric UMAP. Any host
clustering visible here exists in the pretrained representation — no supervision has
touched it.

In [ ]:
seq_id_to_host = dict(zip(metadata_df["seq_id"], metadata_df["host_type"]))
host_labels = [seq_id_to_host[sid] for sid in sequence_embeddings]

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                    random_state=RANDOM_STATE, metric="cosine")
umap_embedding = reducer.fit_transform(np.array(list(sequence_embeddings.values())))

umap_df = pd.DataFrame({
    "UMAP1": umap_embedding[:, 0],
    "UMAP2": umap_embedding[:, 1],
    "host_type": host_labels,
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=umap_df, x="UMAP1", y="UMAP2", hue="host_type", alpha=0.7, s=50)
plt.title("UMAP of frozen ESM-2 embeddings, H5N1 PB2 sequences by host")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---

## 3. Stage 1 — frozen embeddings + classical classifiers

The baseline question: do pretrained ESM-2 embeddings *already* linearly separate host
classes, with no task-specific training at all?

Logistic regression tests linear separability; random forest allows nonlinear axis-aligned
splits. Both use `class_weight="balanced"` (redundant after downsampling, but harmless)
and a stratified 60/20/20 train/val/test split.

In [ ]:
def split_indices(seq_ids, y_encoded, val_size=0.2, test_size=0.2, random_state=RANDOM_STATE):
    """Stratified 60/20/20 split, returned as index arrays over seq_ids."""
    idx = np.arange(len(seq_ids))
    idx_temp, idx_test = train_test_split(
        idx, test_size=test_size, random_state=random_state, stratify=y_encoded)
    idx_train, idx_val = train_test_split(
        idx_temp, test_size=val_size / (1 - test_size),
        random_state=random_state, stratify=y_encoded[idx_temp])
    return idx_train, idx_val, idx_test

In [ ]:
# Save the model weights
import joblib

def train_baseline_classifier(embeddings, metadata_df, val_size=0.2, test_size=0.2,
                              random_state=RANDOM_STATE):
    """Logistic regression and random forest on frozen ESM-2 embeddings."""
    seq_ids = list(embeddings.keys())
    X = np.array([embeddings[sid] for sid in seq_ids])

    label_dict = dict(zip(metadata_df["seq_id"], metadata_df["host_type"]))
    y = np.array([label_dict[sid] for sid in seq_ids])

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    i_tr, i_va, i_te = split_indices(seq_ids, y_encoded, val_size, test_size, random_state)
    X_train, X_val, X_test = X[i_tr], X[i_va], X[i_te]
    y_train, y_val, y_test = y_encoded[i_tr], y_encoded[i_va], y_encoded[i_te]

    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    classifiers = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=random_state, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            n_estimators=100, random_state=random_state, class_weight="balanced"),
    }

    results = {}
    for name, clf in classifiers.items():
        print(f"\n{'=' * 60}\nTraining {name}...")
        clf.fit(X_train, y_train)
        # Save only the random forest -- it is the model we ship. Without this
        # guard every classifier writes to the same path and the last one wins.
        if name == "Random Forest":
            joblib.dump(clf, RESULTS_DIR / "baseline" / "pb2_rf_classifier.joblib")

        y_pred_train, y_pred_val, y_pred = (
            clf.predict(X_train), clf.predict(X_val), clf.predict(X_test))

        print(f"Train acc: {accuracy_score(y_train, y_pred_train):.4f}")
        print(f"Val acc:   {accuracy_score(y_val, y_pred_val):.4f}")
        print(f"Test acc:  {accuracy_score(y_test, y_pred):.4f}\n")
        print(classification_report(y_test, y_pred, target_names=le.classes_))

        results[name] = {
            "model": clf,
            "train_accuracy": accuracy_score(y_train, y_pred_train),
            "val_accuracy": accuracy_score(y_val, y_pred_val),
            "test_accuracy": accuracy_score(y_test, y_pred),
            "y_test": y_test,
            "y_pred": y_pred,
            "confusion_matrix": confusion_matrix(y_test, y_pred),
        }

    return results, le

In [ ]:
def plot_confusion_matrix(cm, classes, title="Confusion Matrix"):
    """Counts in each cell, row-normalised proportion in grey beneath."""
    plt.figure(figsize=(8, 6))
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes,
                cbar_kws={"label": "Count"})

    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j + 0.5, i + 0.7, f"({cm_norm[i, j]:.2f})",
                     ha="center", va="center", fontsize=9, color="gray")

    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    return plt.gcf()

In [ ]:
baseline_results, label_encoder = train_baseline_classifier(sequence_embeddings, metadata_df)

for name, result in baseline_results.items():
    plot_confusion_matrix(result["confusion_matrix"], label_encoder.classes_,
                          title=f"{name} — Test Set")
    plt.show()

### Attention contact maps

ESM-2's contact head predicts residue–residue contacts from attention. This is purely
interpretive — none of it feeds the classifiers — but it lets us ask whether the model's
internal structural picture of PB2 differs by host.

`return_contacts=True` is memory-hungry; this runs on a 100-sequence-per-host subset with
`batch_size=1`.

In [ ]:
def extract_attention(sequences, metadata_df, model, alphabet, device, n_per_host=100):
    """Contact maps for a stratified subset. Visualization only."""
    attention_weights = {}
    seq_lookup = dict(sequences)

    sampled_seqs = []
    for host_type in ["human", "avian", "mammal"]:
        host_seqs = metadata_df.loc[metadata_df["host_type"] == host_type, "seq_id"].values
        chosen = np.random.choice(host_seqs, min(n_per_host, len(host_seqs)), replace=False)
        sampled_seqs.extend((sid, seq_lookup[sid]) for sid in chosen)

    print(f"Extracting attention for {len(sampled_seqs)} sequences...")
    batch_converter = alphabet.get_batch_converter()

    for i, (seq_id, seq) in enumerate(sampled_seqs):
        _, _, batch_tokens = batch_converter([(seq_id, seq)])
        tokens_len = (batch_tokens != alphabet.padding_idx).sum(1)[0].item()
        batch_tokens = batch_tokens.to(device)

        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[33], return_contacts=True)

        attention_weights[seq_id] = results["contacts"][0, 1:tokens_len - 1,
                                                        1:tokens_len - 1].cpu().numpy()
        del batch_tokens, results
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(sampled_seqs)}")

    return attention_weights, sampled_seqs

In [ ]:
def plot_average_contact_maps(attention_weights, sampled_seqs, seq_to_host, output_dir=None):
    """Mean contact map per host class, cropped to the shortest sequence in each group."""
    host_attention = defaultdict(list)
    for seq_id, _ in sampled_seqs:
        host_attention[seq_to_host[seq_id]].append(attention_weights[seq_id])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, host_type in zip(axes, ["human", "avian", "mammal"]):
        attentions = host_attention.get(host_type, [])
        if not attentions:
            ax.axis("off")
            ax.set_title(f"{host_type.title()} (no data)")
            continue

        min_size = min(a.shape[0] for a in attentions)
        avg = np.mean([a[:min_size, :min_size] for a in attentions], axis=0)

        im = ax.imshow(avg, cmap="viridis", aspect="equal")
        ax.set_title(f"{host_type.title()} (n={len(attentions)})")
        ax.set_xlabel("Position")
        ax.set_ylabel("Position")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if output_dir is not None:
        fig.savefig(Path(output_dir) / "average_contact_maps_by_host.png",
                    dpi=300, bbox_inches="tight")
    return fig

In [ ]:
# Optional — slow. Set to True to regenerate the contact-map figures.
RUN_ATTENTION_ANALYSIS = False

if RUN_ATTENTION_ANALYSIS:
    attention_weights, sampled_seqs = extract_attention(
        sequences, metadata_df, model, alphabet, device, n_per_host=100)
    plot_average_contact_maps(attention_weights, sampled_seqs, seq_id_to_host)
    plt.show()

---

## 4. Stage 2 — frozen embeddings + MLP head

Same frozen embeddings, but a learned nonlinear head instead of scikit-learn: a two-layer
MLP (1280 → 512 → 3) with ReLU and dropout, trained with Adam and best-checkpoint
selection on validation accuracy.

In [ ]:
class MLPhead(nn.Module):
    """Two-layer MLP over frozen mean-pooled embeddings."""

    def __init__(self, in_dim, num_classes, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [ ]:
def train_nn_classifier(embeddings, metadata_df, val_size=0.2, test_size=0.2,
                        random_state=RANDOM_STATE, hidden_dim=512, dropout=0.1,
                        lr=1e-3, batch_size=8, num_epochs=20, device=None):
    """Train the MLP head on frozen ESM-2 embeddings."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    seq_ids = list(embeddings.keys())
    X = np.array([embeddings[sid] for sid in seq_ids])
    label_dict = dict(zip(metadata_df["seq_id"], metadata_df["host_type"]))
    y = np.array([label_dict[sid] for sid in seq_ids])

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    i_tr, i_va, i_te = split_indices(seq_ids, y_encoded, val_size, test_size, random_state)
    splits = {}
    for name, idx in [("train", i_tr), ("val", i_va), ("test", i_te)]:
        splits[name] = torch.utils.data.TensorDataset(
            torch.from_numpy(X[idx]).float(), torch.from_numpy(y_encoded[idx]).long())

    print(f"Train: {len(i_tr)}, Val: {len(i_va)}, Test: {len(i_te)}")

    loaders = {
        name: torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=(name == "train"))
        for name, ds in splits.items()
    }

    net = MLPhead(X.shape[1], len(le.classes_), hidden_dim, dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    best_val_acc, best_state = 0.0, None

    for epoch in range(num_epochs):
        net.train()
        running_loss, correct, total = 0.0, 0, 0
        for batch_X, batch_y in loaders["train"]:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = net(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == batch_y).sum().item()
            total += batch_y.size(0)

        net.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for batch_X, batch_y in loaders["val"]:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                correct_val += (net(batch_X).argmax(1) == batch_y).sum().item()
                total_val += batch_y.size(0)
        val_acc = correct_val / total_val

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}

        print(f"Epoch [{epoch + 1}/{num_epochs}], "
              f"Loss: {running_loss / len(loaders['train']):.4f}, "
              f"Train Acc: {correct / total:.4f}, Val Acc: {val_acc:.4f}")

    if best_state is not None:
        net.load_state_dict(best_state)

    def predict(loader):
        net.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch_X, batch_y in loader:
                preds.append(net(batch_X.to(device)).argmax(1).cpu().numpy())
                labels.append(batch_y.numpy())
        return np.concatenate(preds), np.concatenate(labels)

    y_pred, y_true = predict(loaders["test"])
    test_acc = accuracy_score(y_true, y_pred)
    print(f"\nFinal test accuracy: {test_acc:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    return {"NN Head": {
        "model": net,
        "test_accuracy": test_acc,
        "y_test": y_true,
        "y_pred": y_pred,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }}, le

In [ ]:
nn_results, nn_le = train_nn_classifier(sequence_embeddings, metadata_df)

plot_confusion_matrix(nn_results["NN Head"]["confusion_matrix"], nn_le.classes_,
                      title="MLP Head — Test Set")
plt.show()

---

## 5. Stage 3 — fine-tuning ESM-2 end to end

Now we let gradients into the transformer itself.

**Layer freezing.** Training all 650M parameters on 960 examples would overfit instantly
and does not fit in memory at reasonable batch sizes. We freeze the first `freeze_layers`
transformer blocks and fine-tune only the top ones. Two configurations were run: freezing
the first 30 of 33 layers (tuning the last 3) and freezing the first 25 (tuning the last 8).

**Attention pooling.** Instead of mean-pooling residues uniformly, a small learned
attention module scores each residue and takes a weighted sum. The motivation is
biological: host adaptation is driven by a handful of specific positions (627, 701), so a
pooling operation that can concentrate on them should beat an average that dilutes them
across ~759 residues.

**Regularisation.** Deeper head with batch norm and dropout, AdamW with weight decay 0.01,
gradient clipping at 1.0, cosine-annealed LR, and early stopping on validation accuracy.

In [ ]:
class ESM2Dataset(Dataset):
    """Holds raw sequences; tokenization happens in collate_fn."""

    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq_id, seq = self.sequences[idx]
        return seq_id, seq, self.labels[idx]


def collate_fn(batch, batch_converter):
    """Tokenize a batch. Note the class labels are kept separate from ESM's
    internal 'labels' output, which is just the sequence IDs we passed in."""
    seq_ids, seqs, class_labels = zip(*batch)
    _, _, tokens = batch_converter(list(zip(seq_ids, seqs)))
    return tokens, torch.tensor(class_labels, dtype=torch.long)

In [ ]:
class AttentionPooling(nn.Module):
    """Learned attention-weighted pooling over residue embeddings."""

    def __init__(self, embed_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.Tanh(),
            nn.Linear(embed_dim // 2, 1),
        )

    def forward(self, embeddings, mask):
        weights = self.attention(embeddings).squeeze(-1)
        weights = weights.masked_fill(~mask.bool(), float("-inf"))
        weights = F.softmax(weights, dim=1)
        return torch.sum(weights.unsqueeze(-1) * embeddings, dim=1)

In [ ]:
class ESM2FineTuned(nn.Module):
    """ESM-2 with upper layers unfrozen, attention pooling, and a deep classifier head."""

    def __init__(self, num_classes, model_name="esm2_t33_650M_UR50D",
                 freeze_layers=25, dropout=0.2, use_attention_pooling=True):
        super().__init__()
        self.esm, self.alphabet = esm.pretrained.load_model_and_alphabet(model_name)
        self.batch_converter = self.alphabet.get_batch_converter()
        self.use_attention_pooling = use_attention_pooling

        for name, param in self.esm.named_parameters():
            layer_num = self._get_layer_num(name)
            if layer_num is not None and layer_num < freeze_layers:
                param.requires_grad = False

        embed_dim = self.esm.embed_dim
        self.pooling = AttentionPooling(embed_dim) if use_attention_pooling else None

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    @staticmethod
    def _get_layer_num(name):
        if "layers." in name:
            try:
                return int(name.split("layers.")[1].split(".")[0])
            except (IndexError, ValueError):
                return None
        return None

    def forward(self, tokens):
        results = self.esm(tokens, repr_layers=[33], return_contacts=False)
        embeddings = results["representations"][33]
        seq_lens = (tokens != self.alphabet.padding_idx).sum(1)

        batch_size, max_len = tokens.size(0), tokens.size(1)
        mask = torch.zeros(batch_size, max_len, dtype=torch.bool, device=tokens.device)
        for i, seq_len in enumerate(seq_lens):
            if seq_len > 2:
                mask[i, 1:seq_len - 1] = True   # exclude <cls> and <eos>
            else:
                mask[i, :seq_len] = True

        pooled = []
        for i, seq_len in enumerate(seq_lens):
            seq_emb = embeddings[i, 1:seq_len - 1] if seq_len > 2 else embeddings[i, :seq_len]
            if self.use_attention_pooling and seq_emb.size(0) > 1:
                valid_emb = embeddings[i:i + 1, :seq_len]
                seq_mask = mask[i, :seq_len].unsqueeze(0)
                pooled.append(self.pooling(valid_emb, seq_mask).squeeze(0))
            else:
                pooled.append(seq_emb.mean(0))   # fallback

        return self.classifier(torch.stack(pooled))

In [ ]:
def train_esm2_finetuned(sequences, metadata_df, val_size=0.2, test_size=0.2,
                         random_state=RANDOM_STATE, model_name="esm2_t33_650M_UR50D",
                         freeze_layers=25, lr=5e-5, batch_size=8, num_epochs=50,
                         device=None, patience=10, min_delta=0.001):
    """Fine-tune ESM-2 for host classification with early stopping."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    seq_ids = [sid for sid, _ in sequences]
    label_dict = dict(zip(metadata_df["seq_id"], metadata_df["host_type"]))
    y = np.array([label_dict[sid] for sid in seq_ids])

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    seq_temp, seq_test, y_temp, y_test = train_test_split(
        sequences, y_encoded, test_size=test_size,
        random_state=random_state, stratify=y_encoded)
    seq_train, seq_val, y_train, y_val = train_test_split(
        seq_temp, y_temp, test_size=val_size / (1 - test_size),
        random_state=random_state, stratify=y_temp)

    print(f"Train: {len(seq_train)}, Val: {len(seq_val)}, Test: {len(seq_test)}")

    net = ESM2FineTuned(len(le.classes_), model_name, freeze_layers,
                        use_attention_pooling=True).to(device)

    def make_loader(seqs, labels, shuffle):
        return DataLoader(ESM2Dataset(seqs, labels), batch_size=batch_size, shuffle=shuffle,
                          collate_fn=lambda x: collate_fn(x, net.batch_converter))

    train_loader = make_loader(seq_train, y_train, True)
    val_loader   = make_loader(seq_val, y_val, False)
    test_loader  = make_loader(seq_test, y_test, False)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-7)

    best_val_acc, best_state, patience_counter = 0.0, None, 0
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        net.train()
        running_loss, correct, total = 0.0, 0, 0
        for batch_tokens, batch_labels in train_loader:
            batch_tokens, batch_labels = batch_tokens.to(device), batch_labels.to(device)
            optimizer.zero_grad()
            outputs = net(batch_tokens)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == batch_labels).sum().item()
            total += batch_labels.size(0)

        net.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for batch_tokens, batch_labels in val_loader:
                batch_tokens, batch_labels = batch_tokens.to(device), batch_labels.to(device)
                correct_val += (net(batch_tokens).argmax(1) == batch_labels).sum().item()
                total_val += batch_labels.size(0)

        val_acc = correct_val / total_val
        history["train_loss"].append(running_loss / len(train_loader))
        history["train_acc"].append(correct / total)
        history["val_acc"].append(val_acc)
        scheduler.step()

        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch [{epoch + 1}/{num_epochs}], "
              f"Train Loss: {history['train_loss'][-1]:.4f}, "
              f"Train Acc: {history['train_acc'][-1]:.4f}, "
              f"Val Acc: {val_acc:.4f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if patience_counter >= patience:
            print(f"\nEarly stopping after {epoch + 1} epochs")
            break

    if best_state is not None:
        net.load_state_dict(best_state)
        print(f"\nLoaded best checkpoint (val acc {best_val_acc:.4f})")

    def evaluate(loader):
        net.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch_tokens, batch_labels in loader:
                preds.append(net(batch_tokens.to(device)).argmax(1).cpu().numpy())
                labels.append(batch_labels.numpy())
        return np.concatenate(preds), np.concatenate(labels)

    y_pred, y_true = evaluate(test_loader)
    test_acc = accuracy_score(y_true, y_pred)
    print(f"\nFinal test accuracy: {test_acc:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    return {"ESM-2 Fine-tuned": {
        "model": net,
        "test_accuracy": test_acc,
        "y_test": y_true,
        "y_pred": y_pred,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "history": history,
    }}, le

In [ ]:
# Expensive: ~16GB VRAM, tens of minutes to hours depending on GPU.
RUN_FINETUNING = False

if RUN_FINETUNING:
    ft_results, ft_le = train_esm2_finetuned(
        sequences, metadata_df,
        freeze_layers=30,   # tune only the last 3 of 33 layers
        lr=1e-4,
        batch_size=4,
        num_epochs=10,
    )

    plot_confusion_matrix(ft_results["ESM-2 Fine-tuned"]["confusion_matrix"],
                          ft_le.classes_, title="ESM-2 Fine-tuned — Test Set")
    plt.show()

    history = ft_results["ESM-2 Fine-tuned"]["history"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], marker="o")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("Epoch")
    axes[1].plot(history["train_acc"], marker="o", label="train")
    axes[1].plot(history["val_acc"], marker="s", label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

---

## 6. Results and discussion

| Stage | Model | What trains | Test accuracy |
|---|---|---|---|
| 1 | Frozen ESM-2 + logistic regression | classifier only | 83.80% |
| 1 | **Frozen ESM-2 + random forest** | classifier only | **90.97%** |
| 2 | Frozen ESM-2 + MLP head | 2-layer MLP | 84.11% |
| 3 | Fine-tuned ESM-2 (freeze 30, tune 3) | top 3 layers + head | 89.10% |

Chance is 33.3% on the balanced three-class problem.

### The headline finding is a negative result

Fine-tuning **did not beat** frozen features. A random forest over embeddings that ESM-2
produced without ever seeing an influenza label outperformed a network in which we
unfroze 3 transformer layers, added attention pooling, and trained a four-layer head with
cosine annealing and early stopping.

Three explanations, in rough order of how much weight they deserve:

1. **The dataset is far too small for fine-tuning.** 960 training sequences against tens
   of millions of trainable parameters. Even three unfrozen ESM-2 blocks is ~60M
   parameters. The fine-tuning curves are consistent with this: training accuracy plateaus
   near 0.82 for seven straight epochs before finally moving, which reads as an
   underfit-then-memorise pattern rather than healthy learning.

2. **The host signal is concentrated in a few residues, and mean-pooled frozen embeddings
   already capture it.** PB2 host adaptation is substantially driven by positions 627 and
   701. If the pretrained representation of those residues already differs by host, a
   random forest can find that in the pooled vector, and there is little left for
   fine-tuning to add.

3. **Random forest suits this regime.** With 1602 samples and 1280 features, an ensemble
   of axis-aligned splits is well matched to the sample-to-dimension ratio in a way that
   gradient-trained deep heads are not.

### Caveats worth stating plainly

- **Single split, single seed.** Every number above comes from one stratified 60/20/20
  split at `random_state=42`. With 321 test sequences, the difference between 89.10% and
  90.97% is about six sequences — well inside what seed variation could produce. These
  models are **not** distinguishable without cross-validation.
- **No sequence-identity deduplication.** Influenza sequences from the same outbreak are
  near-identical. If highly similar sequences straddle the train/test boundary, accuracy
  is inflated for every model here. This is the single biggest threat to the results, and
  clustering at e.g. 95% identity before splitting is the first thing to fix.
- **Truncated sequences retained.** Partial submissions as short as 32 residues remain in
  the dataset (section 1), and truncation rate may correlate with host class.
- **H5N1 only.** Nothing here shows the models generalise to other subtypes.
- **"Mammal" is a heterogeneous class** spanning dairy cattle, swine, felids, and marine
  mammals with very different adaptation histories.

### Next steps

1. 5-fold stratified cross-validation with multiple seeds, reporting mean ± std. Without
   this the ranking above is not trustworthy.
2. Deduplicate with CD-HIT or MMseqs2 at 95% identity and re-split by cluster.
3. Position-level interpretation: does attention-pooling weight concentrate on residues
   627 and 701? That would be the paper's most interesting figure and it is one plot away.
4. A trivial-baseline check — how far does one-hot encoding of positions 627 and 701 alone
   get you? If that reaches 85%, the language model is contributing much less than the
   headline numbers suggest.
5. Filter sequences below ~700 residues and re-run.

### Data availability

Sequences were obtained from **GISAID EpiFlu** and are not redistributed in this
repository, and no real accessions are included. `data/sample_accessions.csv` documents
the expected input format with synthetic records; see `data/README.md`
for retrieval instructions. We gratefully acknowledge the originating and submitting
laboratories.